# Exercise 007

<a href="https://colab.research.google.com/github/FAIRChemistry/PythonProgramming2025/blob/master/exercises/Exercise007.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Please execute this cell to download the necessary data
!wget https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/scripts/utils.py
!wget https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/data/single_sequence.fasta

from utils import CODON_TABLE, to_triplets

--2025-06-30 13:48:14--  https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/scripts/utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1411 (1.4K) [text/plain]
Saving to: ‘utils.py’

utils.py            100%[===================>]   1.38K  --.-KB/s    in 0s      

2025-06-30 13:48:14 (21.5 MB/s) - ‘utils.py’ saved [1411/1411]

--2025-06-30 13:48:14--  https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/data/single_sequence.fasta
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 877 [text/p

# DNASequence class

Construct a `DNASequence` class that contains the following attributes:

* `id`
* `sequence`
* `organism`
* `gc_content`
* `length`
* `reverse_complement`

Next, implement methods for your class that perform the following tasks:

* `to_amino_acid`: Converts the nucelic acid sequence to an amino acid sequence.
* `align`: Takes another sequence and aligns it against the instance sequence.
* `__repr__`: Define how the contents of your class should be printed.
* `from_fasta`: Define a classmethod that parses a single FASTA entry into your class.

Demonstrate your class by parsing the `single_sequence.fasta` file either manually or via the `from_fasta`-classmethod.

**Tips**

> * Feel free to use the `get_identity`-function of the previous exercise.
> * When implementing the `classmethod` make sure to check if the format is correct. We have so far followed the `>[Header]\n[Sequence]` format.
> * Translate your sequence using the supported `to_triplets` function and `CODON_TABLE` dictionary.
> * Not familiar with reverse complements? Find more info [here](http://genewarrior.com/docs/exp_revcomp.jsp)
> * Dont hesitate using the `dataclass` decorator. It can help you in some ways already. Learn more on how to implement `__post_init__` to maximize customizability [here](https://docs.python.org/3/library/dataclasses.html#post-init-processing)
> * Python lacks type validation and thus you do have limited control of what flows into your class. [PyDantic](https://docs.pydantic.dev/latest/) is an excellent tool to solve this and other issues. Try it out to make your life easier!

In [19]:
#Reusing some code from previous exercises
%pip install biopython
from Bio import pairwise2


def get_identity(seq1: str, seq2: str):
    """Aligns two sequences using BioPython

    Args:
        seq1 (str): Query sequence to align to
        seq2 (str): Target sequence to align with

    Returns:
        float: Identity of the resulting alignment

    """
    return pairwise2.align.globalxx(seq1, seq2, score_only=True) / len(seq1)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.9 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [66]:
#Standard classes
#Reusing some code from previous exercises

class DNASequence:
  def __init__(self, id, organism, sequence):
    self.id = id
    self.organism = organism
    self.sequence = sequence.upper()

    self.length = len(self.sequence)
    self.gc_content = (self.sequence.count("G")+self.sequence.count("C"))/self.length

  #Adding reverse_complement attribute
    complement = ""
    complements_dict= {"A" : "T", "T" : "A", "G" : "C", "C":"G"}
    for nucleotide in self.sequence:
      complement += complements_dict.get(nucleotide)
    self.reverse_complement = complement[::-1]

  #Adding new methods
  def to_amino_acid(self):
    seq=str(self.sequence) # Use self.sequence
    triplets=to_triplets(seq)
    aa_string=""
    for triplet in triplets:
      aa=CODON_TABLE.get(triplet)
      if aa == "_" or aa == None:
        continue
      else:
        aa_string=aa_string+aa
    return(aa_string)

  def align(self, other):
    assert (type(self)==type(other)), (f"Only types of DNASequence can be used for comparison. Got '{type(other)}', which is invalid.")
    return get_identity(self.sequence, other.sequence)

  def __repr__(self):
    repr_string = ""
    for key,value in self.__dict__.items():
      if value is not None: # Check if value is not None
        repr_string += f"{key}: {value}\n"
    return repr_string

  @classmethod
  def from_fasta(cls, filename):
    """Reads FASTA file with a single sequence"""
    r= open(f"{filename}", "r").readlines()
    for i, line in enumerate(r):
      if not i % 2:
        organism, id = line.lstrip(">").split("|")
        continue
      else:
        obj = DNASequence(
          id=id.strip(),
          organism=organism.strip(),
          sequence=line.strip())
    return obj

In [68]:
#Standard class instance
#Demonstrate your class by parsing the single_sequence.fasta file either manually or via the from_fasta-classmethod.
file="single_sequence.fasta"
instance=DNASequence.from_fasta(file)
print(instance)
print(instance.to_amino_acid())

id: 1
organism: ecoli
sequence: ATGCGTTCTCGCTATTTGTTACATCAATATTTTGTTCAGGTACAGTTTGCAGCGCCGTCGCCAGCGCCAACGGATTCCATGTCATATATTATTCCATATAGATTAAGTTTAAATATTAATAAAATGAATATTTGCAATACGTAATTATCTTACCAGCTATAGACAAAAAAAAACCATCCAAATCTGGATGGCTTTTCATAATTCAGAGGAACTAGCTGCGCTGACGAACCGCTTCAAATAAGCAAATTCCGGTTGCAACCGAAACGTTCAGGGAAGAAACACTTCCTGCCATTGGGATGCTGATCAACTCATCGCAATGTTCACGGGTCAGGCGACGCATACCTTCACCTTCCGCGCCCATCACCAGCGCCAGGCGTCCGGTCATTTTGCTTTGATAGAGCGTATGATCCGCCTCACCTGCCGTACCGACGATCCAGATATTCTCTTCCTGCAACATACGCATGGTGCGCGCAAGGTTAGTCACCCGAATCAGTGGAACGCTTTCTGCCGCGCCGCAGGCTACTTTTTTCGCCGTGGCGTTGAGCTGTGCGGAGCGATCTTTCGGCACAATCACCGCGTGAACGCCAGCAGCGTCCGCGCTACGCAGGCACGCGCCGAGGTTGTGCGGATCGGTTACACCGTCGAGGATCAGCAGGAACGGTTGATCGAGCGAAGCGATCAGATCCGGCAGATCGTTTTCCTGGTACTGACGTCCTGGCTTCACGCGGGCGATAATGCCCTGATGCACGGCACCGTCGCTTTTCTCGTCGAGATATTGGCGGTTTGCCAACTGGATAACCACGCCCTGGGACTCAAGGGCGTGGATCAGCGGTAACAGACGTTTATCTTCACGGCCTTTTAAAATAA
length: 867
gc_content: 0.5074971164936563
reverse_complement: TTATTTTAAAAGGCCGTGAAGATAAACGTCTGTTACC

In [ ]:
#Using dataclasses
from dataclasses import dataclass

@dataclass
class DNASequencedc:
  id: str
  organism: str
  sequence: str
  length: int
  gc_content: float

In [ ]:
#Dataclass instace